In [ ]:
import io
import os
import pandas as pd
from google.colab import files
from IPython.display import display, HTML

# ===========================================================
# 1. LOAD EXCEL OR CSV FILE
# ===========================================================
file_name = "/content/Excel (6).xlsx"
if os.path.exists(file_name):
    if file_name.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(file_name)
    else:
        df = pd.read_csv(file_name)

    # ========================================================
    # 2. STUDENT INFORMATION
    # ========================================================
    student_name = "Sheena"
    course = "BSCS"
    year_level = "1"

    # ========================================================
    # 3. CLEAN AND REORDER THE DATA
    # ========================================================
    df.columns = df.columns.str.strip()
    df["Units"] = pd.to_numeric(df["Units"], errors="coerce")
    df["Grade"] = pd.to_numeric(df["Grade"], errors="coerce")
    grades_df = df.dropna(subset=["Grade"]).copy()

    # Reorder columns: Put Units between Teachers and Grade
    ordered_cols = ['Code', 'Description', 'Teachers', 'Units', 'Grade']
    final_cols = [c for c in ordered_cols if c in grades_df.columns]
    for c in grades_df.columns:
        if c not in final_cols:
            final_cols.append(c)
    grades_df = grades_df[final_cols]

    # ========================================================
    # 4. CALCULATE SUMMARY
    # ========================================================
    total_subjects = len(grades_df)
    total_units = grades_df["Units"].sum() if "Units" in grades_df.columns else 0
    average_grade = grades_df["Grade"].mean() if "Grade" in grades_df.columns else 0
    gwa = (
        (grades_df["Grade"] * grades_df["Units"]).sum() / total_units
        if total_units > 0 else 0
    )

    # ========================================================
    # 5. GENERATE THE UNIFIED SINGLE TABLE HTML
    # ========================================================

    # Generate individual table rows for subjects
    rows_html = ""
    for i, row in grades_df.iterrows():
        row_class = "even-row" if i % 2 == 0 else "odd-row"
        rows_html += f"""
        <tr class="{row_class}">
            <td class="text-center font-bold">{row.get('Code', '')}</td>
            <td class="text-left">{row.get('Description', '')}</td>
            <td class="text-left">{row.get('Teachers', '')}</td>
            <td class="text-center">{row.get('Units', '')}</td>
            <td class="text-center font-bold">{row.get('Grade', ''):.2f}</td>
        </tr>
        """

    # CSS tailored with pure white base properties:
    html = f"""
    <style>
    /* Unified design for both Screen and PDF with clean white backgrounds */
    .report-container {{
        width: 100%;
        max-width: 100%;
        margin: 0;
        padding: 10px;
        font-family: Arial, sans-serif;
        color: #333;
        background-color: #ffffff;
        box-sizing: border-box;
    }}

    /* UNIFIED TABLE STYLE */
    .unified-table {{
        width: 100%;
        border-collapse: collapse;
        border: 2px solid #afb9b4;
        border-radius: 12px;
        overflow: hidden;
        background-color: #ffffff;
    }}

    /* MAIN HEADER */
    .main-header {{
        background: #ffffff;
        color: #2b3a33;
        text-align: center;
        font-size: 24px;
        font-weight: bold;
        padding: 20px;
        letter-spacing: 1px;
        border-bottom: 2px solid #afb9b4;
    }}

    /* STUDENT INFO HEADER CELL */
    .student-info-cell {{
        background: #ffffff;
        padding: 15px 20px;
        border-bottom: 2px solid #cbcbcb;
    }}

    .info-grid {{
        display: grid;
        grid-template-columns: repeat(3, 1fr);
        gap: 15px;
        font-size: 15px;
    }}

    .info-item {{
        display: flex;
        gap: 8px;
    }}

    .info-label {{
        font-weight: bold;
        color: #555555;
    }}

    /* COLUMN HEADERS */
    .col-header th {{
        background: #ffffff;
        color: #2b3a33;
        padding: 12px;
        font-weight: bold;
        font-size: 14px;
        border-bottom: 2px solid #afb9b4;
    }}

    /* TABLE BODY */
    .unified-table td {{
        padding: 12px 15px;
        border-bottom: 1px solid #e3e2e3;
        font-size: 14px;
        background-color: #ffffff;
    }}

    .even-row {{
        background-color: #ffffff;
    }}

    .odd-row {{
        background-color: #ffffff;
    }}

    .unified-table tr:hover td {{
        background-color: #fcfcfc;
    }}

    /* SUMMARY SECTION AT THE BOTTOM */
    .summary-footer-cell {{
        background: #ffffff;
        border-top: 2px solid #cbcbcb;
        padding: 15px 20px;
    }}

    .summary-container {{
        display: flex;
        justify-content: space-between;
        align-items: center;
        flex-wrap: wrap;
        gap: 15px;
    }}

    .summary-item {{
        font-size: 15px;
        white-space: nowrap;
    }}

    .summary-label {{
        font-weight: bold;
        color: #444444;
    }}

    .summary-value {{
        font-weight: bold;
        color: #222222;
    }}

    /* HIGHLIGHTED GWA BOX */
    .gwa-highlight {{
        background: #ffffff;
        color: #2b3a33;
        border: 2px solid #afb9b4;
        padding: 8px 18px;
        border-radius: 6px;
        font-weight: bold;
        display: inline-flex;
        gap: 8px;
    }}

    /* ALIGNMENTS & UTILITIES */
    .text-center {{
        text-align: center !important;
    }}
    .text-left {{
        text-align: left !important;
    }}
    .font-bold {{
        font-weight: bold;
    }}

    /* BUTTON STYLES */
    .btn-container {{
        text-align: center;
        margin-bottom: 20px;
    }}

    .print-btn {{
        background-color: #ffffff;
        color: #2b3a33;
        border: 1px solid #999999;
        padding: 12px 25px;
        font-size: 16px;
        font-weight: bold;
        border-radius: 30px;
        cursor: pointer;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.05);
        transition: background-color 0.2s;
    }}

    .print-btn:hover {{
        background-color: #afb9b4;
        color: #2b3a33;
    }}

    /* PRINT MEDIA RULES */
    @media print {{
        @page {{
            size: auto;
            margin: 0mm !important;
        }}

        body {{
            margin: 0 !important;
            padding: 10px !important;
            background-color: #ffffff !important;
        }}

        .no-print {{
            display: none !important;
        }}

        .report-container {{
            padding: 0 !important;
            width: 100% !important;
            max-width: 100% !important;
        }}

        .unified-table {{
            width: 100% !important;
            border: 2px solid #afb9b4 !important;
            background-color: #ffffff !important;
        }}
    }}
    </style>

    <div class="report-container">
        <div class="btn-container no-print">
            <button class="print-btn" onclick="var oldTitle = document.title; document.title = ''; window.print(); setTimeout(function() {{ document.title = oldTitle; }}, 100);">
                🖨️ Save / Print to PDF
            </button>
        </div>

        <table class="unified-table">
            <thead>
                <!-- 1. Report Title Row -->
                <tr>
                    <th colspan="5" class="main-header">
                        STUDENT GRADE REPORT
                    </th>
                </tr>
                <!-- 2. Student Info Metadata Row -->
                <tr>
                    <td colspan="5" class="student-info-cell">
                        <div class="info-grid">
                            <div class="info-item">
                                <span class="info-label">Student Name:</span>
                                <span>{student_name}</span>
                            </div>
                            <div class="info-item">
                                <span class="info-label">Course:</span>
                                <span>{course}</span>
                            </div>
                            <div class="info-item">
                                <span class="info-label">Year Level:</span>
                                <span>{year_level}</span>
                            </div>
                        </div>
                    </td>
                </tr>
                <!-- 3. Grades Columns Row -->
                <tr class="col-header">
                    <th style="width: 15%;">Code</th>
                    <th style="width: 35%;">Description</th>
                    <th style="width: 25%;">Teachers</th>
                    <th style="width: 12%;">Units</th>
                    <th style="width: 13%;">Grade</th>
                </tr>
            </thead>
            <tbody>
                <!-- 4. Dynamic Grade Rows -->
                {rows_html}

                <!-- 5. Unified Footer Summary Row -->
                <tr>
                    <td colspan="5" class="summary-footer-cell">
                        <div class="summary-container">
                            <div class="summary-item">
                                <span class="summary-label">Total Subjects:</span>
                                <span class="summary-value">{total_subjects}</span>
                            </div>
                            <div class="summary-item">
                                <span class="summary-label">Total Units:</span>
                                <span class="summary-value">{total_units:.1f}</span>
                            </div>
                            <div class="summary-item">
                                <span class="summary-label">Average Grade:</span>
                                <span class="summary-value">{average_grade:.2f}</span>
                            </div>
                            <div class="summary-item gwa-highlight">
                                <span style="color: #2b3a33;">GWA:</span>
                                <span style="color: #2b3a33;">{gwa:.4f}</span>
                            </div>
                        </div>
                    </td>
                </tr>
            </tbody>
        </table>
    </div>
    """
    display(HTML(html))
else:
    print(f"File {file_name} not found.")

In [ ]:
# ===========================================================
# 5. GENERATE THE UNIFIED SINGLE TABLE HTML
# ===========================================================

rows_html = ""

for i, row in grades_df.iterrows():
    row_class = "even-row" if i % 2 == 0 else "odd-row"

    # Safely format values
    code = row.get("Code", "")
    description = row.get("Description", "")
    teachers = row.get("Teachers", "")
    units = row.get("Units", "")
    grade = row.get("Grade", "")

    units_display = f"{units:.1f}" if pd.notna(units) else ""
    grade_display = f"{grade:.2f}" if pd.notna(grade) else ""

    rows_html += f"""
    <tr class="{row_class}">
        <td>{code}</td>
        <td>{description}</td>
        <td>{teachers}</td>
        <td>{units_display}</td>
        <td>{grade_display}</td>
    </tr>
    """

# ===========================================================
# HTML + CSS
# ===========================================================

html = f"""
<style>

* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    padding: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #ffffff;
    color: #111111;
}}

.report-container {{
    width: 100%;
    max-width: 1200px;
    margin: 0 auto;
    padding: 15px;
    background: #ffffff;
}}

.btn-container {{
    text-align: center;
    margin-bottom: 20px;
}}

.print-btn {{
    background-color: #ffffff;
    color: #111111;
    border: 1px solid #111111;
    padding: 12px 25px;
    font-size: 16px;
    font-weight: bold;
    border-radius: 30px;
    cursor: pointer;
}}

.print-btn:hover {{
    background-color: #111111;
    color: #ffffff;
}}

/* ==========================================================
   MAIN TABLE
   ========================================================== */

.unified-table {{
    width: 100%;
    table-layout: fixed;
    border-collapse: collapse;
    border: 2px solid #111111;
    background-color: #ffffff;
    text-align: center;
}}

/* ==========================================================
   TITLE
   ========================================================== */

.main-header {{
    background: #ffffff;
    color: #111111;
    text-align: center !important;
    vertical-align: middle !important;
    font-size: 24px;
    font-weight: bold;
    padding: 20px;
    letter-spacing: 1.5px;
    border-bottom: 2px solid #111111;
}}

/* ==========================================================
   STUDENT INFORMATION
   ========================================================== */

.student-info-cell {{
    background: #ffffff;
    padding: 15px 20px;
    border-bottom: 2px solid #111111;
    text-align: center !important;
    vertical-align: middle !important;
}}

.info-grid {{
    width: 100%;
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 15px;
    align-items: center;
    justify-items: center;
    text-align: center;
    font-size: 15px;
}}

.info-item {{
    width: 100%;
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 8px;
    text-align: center;
}}

.info-label {{
    font-weight: bold;
    color: #111111;
}}

/* ==========================================================
   COLUMN HEADERS
   ========================================================== */

.col-header th {{
    background: #ffffff;
    color: #111111;
    padding: 12px 8px;
    font-weight: bold;
    font-size: 14px;
    text-align: center !important;
    vertical-align: middle !important;
    border-bottom: 2px solid #111111;
    border-right: 1px solid #cccccc;
}}

.col-header th:last-child {{
    border-right: none;
}}

/* ==========================================================
   TABLE BODY
   ========================================================== */

.unified-table tbody td {{
    padding: 12px 10px;
    font-size: 14px;
    background-color: #ffffff;
    text-align: center !important;
    vertical-align: middle !important;
    border-bottom: 1px solid #dddddd;
    border-right: 1px solid #dddddd;
    word-wrap: break-word;
    overflow-wrap: break-word;
}}

.unified-table tbody td:last-child {{
    border-right: none;
}}

.even-row td,
.odd-row td {{
    background-color: #ffffff !important;
}}

.unified-table tbody tr:hover td {{
    background-color: #f8f8f8 !important;
}}

/* ==========================================================
   SUMMARY
   ========================================================== */

.summary-footer-cell {{
    background: #ffffff;
    border-top: 2px solid #111111;
    padding: 18px 20px;
    text-align: center !important;
    vertical-align: middle !important;
}}

.summary-container {{
    width: 100%;
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 15px;
    align-items: center;
    justify-items: center;
    text-align: center;
}}

.summary-item {{
    width: 100%;
    font-size: 15px;
    text-align: center;
    white-space: nowrap;
}}

.summary-label {{
    font-weight: bold;
    color: #111111;
}}

.summary-value {{
    font-weight: bold;
    color: #111111;
}}

.gwa-highlight {{
    background: #ffffff;
    color: #111111;
    border: 2px solid #111111;
    padding: 8px 18px;
    border-radius: 6px;
    font-weight: bold;
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 8px;
    width: auto;
    min-width: 130px;
}}

/* ==========================================================
   PRINT / PDF
   ========================================================== */

@media print {{

    @page {{
        size: A4 landscape;
        margin: 8mm;
    }}

    html,
    body {{
        width: 100%;
        margin: 0 !important;
        padding: 0 !important;
        background: #ffffff !important;
    }}

    .no-print {{
        display: none !important;
    }}

    .report-container {{
        width: 100% !important;
        max-width: none !important;
        margin: 0 !important;
        padding: 0 !important;
    }}

    .unified-table {{
        width: 100% !important;
        table-layout: fixed !important;
        border: 2px solid #111111 !important;
        border-collapse: collapse !important;
    }}

    .unified-table th,
    .unified-table td {{
        text-align: center !important;
        vertical-align: middle !important;
    }}

    .main-header {{
        text-align: center !important;
    }}

    .student-info-cell {{
        text-align: center !important;
    }}

    .info-grid {{
        display: grid !important;
        grid-template-columns: repeat(3, 1fr) !important;
        text-align: center !important;
    }}

    .info-item {{
        justify-content: center !important;
        text-align: center !important;
    }}

    .summary-container {{
        display: grid !important;
        grid-template-columns: repeat(4, 1fr) !important;
        text-align: center !important;
    }}

    .summary-item {{
        text-align: center !important;
    }}

    .gwa-highlight {{
        justify-content: center !important;
        text-align: center !important;
    }}

    /*
       Prevent table rows from splitting between pages
    */
    tr {{
        page-break-inside: avoid !important;
    }}

    thead {{
        display: table-header-group;
    }}

    tbody {{
        display: table-row-group;
    }}
}}

</style>

<div class="report-container">

    <!-- PRINT BUTTON -->
    <div class="btn-container no-print">
        <button
            class="print-btn"
            onclick="
                var oldTitle = document.title;
                document.title = '';
                window.print();
                setTimeout(function() {{
                    document.title = oldTitle;
                }}, 100);
            "
        >
            🖨️ Save / Print to PDF
        </button>
    </div>

    <!-- =====================================================
         SINGLE UNIFIED TABLE
         ===================================================== -->

    <table class="unified-table">

        <colgroup>
            <col style="width: 15%;">
            <col style="width: 35%;">
            <col style="width: 25%;">
            <col style="width: 12%;">
            <col style="width: 13%;">
        </colgroup>

        <thead>

            <!-- TITLE -->
            <tr>
                <th colspan="5" class="main-header">
                    STUDENT GRADE REPORT
                </th>
            </tr>

            <!-- STUDENT INFORMATION -->
            <tr>
                <td colspan="5" class="student-info-cell">

                    <div class="info-grid">

                        <div class="info-item">
                            <span class="info-label">Student Name:</span>
                            <span>{student_name}</span>
                        </div>

                        <div class="info-item">
                            <span class="info-label">Course:</span>
                            <span>{course}</span>
                        </div>

                        <div class="info-item">
                            <span class="info-label">Year Level:</span>
                            <span>{year_level}</span>
                        </div>

                    </div>

                </td>
            </tr>

            <!-- COLUMN HEADERS -->
            <tr class="col-header">

                <th>Code</th>
                <th>Description</th>
                <th>Teachers</th>
                <th>Units</th>
                <th>Grade</th>

            </tr>

        </thead>

        <tbody>

            <!-- GRADES -->
            {rows_html}

            <!-- SUMMARY -->
            <tr>

                <td colspan="5" class="summary-footer-cell">

                    <div class="summary-container">

                        <div class="summary-item">
                            <span class="summary-label">
                                Total Subjects:
                            </span>
                            <span class="summary-value">
                                {total_subjects}
                            </span>
                        </div>

                        <div class="summary-item">
                            <span class="summary-label">
                                Total Units:
                            </span>
                            <span class="summary-value">
                                {total_units:.1f}
                            </span>
                        </div>

                        <div class="summary-item">
                            <span class="summary-label">
                                Average Grade:
                            </span>
                            <span class="summary-value">
                                {average_grade:.2f}
                            </span>
                        </div>

                        <div class="summary-item gwa-highlight">
                            <span>GWA:</span>
                            <span>{gwa:.4f}</span>
                        </div>

                    </div>

                </td>

            </tr>

        </tbody>

    </table>

</div>
"""

display(HTML(html))


In [ ]:
# ===========================================================
# 5. GENERATE THE UNIFIED SINGLE TABLE HTML
# ===========================================================

rows_html = ""

for i, row in grades_df.iterrows():
    row_class = "even-row" if i % 2 == 0 else "odd-row"

    code = row.get("Code", "")
    description = row.get("Description", "")
    teachers = row.get("Teachers", "")
    units = row.get("Units", "")
    grade = row.get("Grade", "")

    units_display = f"{units:.1f}" if pd.notna(units) else ""
    grade_display = f"{grade:.2f}" if pd.notna(grade) else ""

    rows_html += f"""
    <tr class="{row_class}">
        <td>{code}</td>
        <td>{description}</td>
        <td>{teachers}</td>
        <td>{units_display}</td>
        <td>{grade_display}</td>
    </tr>
    """

# ===========================================================
# HTML + CSS
# ===========================================================

html = f"""
<style>

* {{
    box-sizing: border-box;
}}

body {{
    margin: 0;
    padding: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #ffffff;
    color: #111111;
}}

.report-container {{
    width: 100%;
    max-width: 1200px;
    margin: 0 auto;
    padding: 15px;
    background: #ffffff;
}}

/* ===========================================================
   PRINT BUTTON
   =========================================================== */

.btn-container {{
    text-align: center;
    margin-bottom: 20px;
}}

.print-btn {{
    background-color: #ffffff;
    color: #111111;
    border: 1px solid #111111;
    padding: 12px 25px;
    font-size: 16px;
    font-weight: bold;
    border-radius: 30px;
    cursor: pointer;
}}

.print-btn:hover {{
    background-color: #111111;
    color: #ffffff;
}}

/* ===========================================================
   MAIN TABLE
   =========================================================== */

.unified-table {{
    width: 100%;
    table-layout: fixed;
    border-collapse: collapse;
    border: 2px solid #111111;
    background-color: #ffffff;
}}

/* ===========================================================
   TITLE
   =========================================================== */

.main-header {{
    background: #ffffff;
    color: #111111;
    text-align: left !important;
    vertical-align: middle !important;
    font-size: 24px;
    font-weight: bold;
    padding: 20px;
    letter-spacing: 1.5px;
    border-bottom: 2px solid #111111;
}}

/* ===========================================================
   STUDENT INFORMATION
   =========================================================== */

.student-info-cell {{
    background: #ffffff;
    padding: 15px 20px;
    border-bottom: 2px solid #111111;
    text-align: left !important;
    vertical-align: middle !important;
}}

.info-grid {{
    width: 100%;
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 15px;
    align-items: center;
    text-align: left;
}}

.info-item {{
    width: 100%;
    display: flex;
    justify-content: flex-start;
    align-items: center;
    gap: 8px;
    text-align: left;
}}

.info-label {{
    font-weight: bold;
    color: #111111;
}}

/* ===========================================================
   COLUMN HEADERS
   =========================================================== */

.col-header th {{
    background: #ffffff;
    color: #111111;
    padding: 12px 10px;
    font-weight: bold;
    font-size: 14px;
    text-align: left !important;
    vertical-align: middle !important;
    border-bottom: 2px solid #111111;
    border-right: 1px solid #cccccc;
}}

.col-header th:last-child {{
    border-right: none;
}}

/* ===========================================================
   TABLE BODY
   =========================================================== */

.unified-table tbody td {{
    padding: 12px 10px;
    font-size: 14px;
    background-color: #ffffff;
    text-align: left !important;
    vertical-align: middle !important;
    border-bottom: 1px solid #dddddd;
    border-right: 1px solid #dddddd;
    word-wrap: break-word;
    overflow-wrap: break-word;
}}

.unified-table tbody td:last-child {{
    border-right: none;
}}

.even-row td,
.odd-row td {{
    background-color: #ffffff !important;
}}

.unified-table tbody tr:hover td {{
    background-color: #f8f8f8 !important;
}}

/* ===========================================================
   SUMMARY
   =========================================================== */

.summary-footer-cell {{
    background: #ffffff;
    border-top: 2px solid #111111;
    padding: 18px 20px;
    text-align: left !important;
    vertical-align: middle !important;
}}

.summary-container {{
    width: 100%;
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 15px;
    align-items: center;
    text-align: left;
}}

.summary-item {{
    width: 100%;
    font-size: 15px;
    text-align: left;
    white-space: nowrap;
}}

.summary-label {{
    font-weight: bold;
    color: #111111;
}}

.summary-value {{
    font-weight: bold;
    color: #111111;
}}

.gwa-highlight {{
    background: #ffffff;
    color: #111111;
    border: 2px solid #111111;
    padding: 8px 18px;
    border-radius: 6px;
    font-weight: bold;
    display: flex;
    justify-content: flex-start;
    align-items: center;
    gap: 8px;
    width: fit-content;
}}

/* ===========================================================
   PRINT / PDF
   =========================================================== */

@media print {{

    @page {{
        size: A4 landscape;
        margin: 8mm;
    }}

    html,
    body {{
        width: 100%;
        margin: 0 !important;
        padding: 0 !important;
        background: #ffffff !important;
    }}

    .no-print {{
        display: none !important;
    }}

    .report-container {{
        width: 100% !important;
        max-width: none !important;
        margin: 0 !important;
        padding: 0 !important;
    }}

    .unified-table {{
        width: 100% !important;
        table-layout: fixed !important;
        border: 2px solid #111111 !important;
        border-collapse: collapse !important;
    }}

    /* Force everything to the LEFT in PDF */
    .unified-table th,
    .unified-table td {{
        text-align: left !important;
        vertical-align: middle !important;
    }}

    .main-header {{
        text-align: left !important;
    }}

    .student-info-cell {{
        text-align: left !important;
    }}

    .info-grid {{
        display: grid !important;
        grid-template-columns: repeat(3, 1fr) !important;
        text-align: left !important;
    }}

    .info-item {{
        justify-content: flex-start !important;
        text-align: left !important;
    }}

    .summary-container {{
        display: grid !important;
        grid-template-columns: repeat(4, 1fr) !important;
        text-align: left !important;
    }}

    .summary-item {{
        text-align: left !important;
    }}

    .gwa-highlight {{
        justify-content: flex-start !important;
        text-align: left !important;
    }}

    tr {{
        page-break-inside: avoid !important;
    }}

    thead {{
        display: table-header-group;
    }}

    tbody {{
        display: table-row-group;
    }}
}}

</style>

<div class="report-container">

    <!-- PRINT BUTTON -->
    <div class="btn-container no-print">
        <button
            class="print-btn"
            onclick="
                var oldTitle = document.title;
                document.title = '';
                window.print();
                setTimeout(function() {{
                    document.title = oldTitle;
                }}, 100);
            "
        >
            🖨️ Save / Print to PDF
        </button>
    </div>

    <!-- =====================================================
         SINGLE UNIFIED TABLE
         ===================================================== -->

    <table class="unified-table">

        <colgroup>
            <col style="width: 15%;">
            <col style="width: 35%;">
            <col style="width: 25%;">
            <col style="width: 12%;">
            <col style="width: 13%;">
        </colgroup>

        <thead>

            <!-- TITLE -->
            <tr>
                <th colspan="5" class="main-header">
                    STUDENT GRADE REPORT
                </th>
            </tr>

            <!-- STUDENT INFORMATION -->
            <tr>
                <td colspan="5" class="student-info-cell">

                    <div class="info-grid">

                        <div class="info-item">
                            <span class="info-label">Student Name:</span>
                            <span>{student_name}</span>
                        </div>

                        <div class="info-item">
                            <span class="info-label">Course:</span>
                            <span>{course}</span>
                        </div>

                        <div class="info-item">
                            <span class="info-label">Year Level:</span>
                            <span>{year_level}</span>
                        </div>

                    </div>

                </td>
            </tr>

            <!-- COLUMN HEADERS -->
            <tr class="col-header">
                <th>Code</th>
                <th>Description</th>
                <th>Teachers</th>
                <th>Units</th>
                <th>Grade</th>
            </tr>

        </thead>

        <tbody>

            <!-- GRADES -->
            {rows_html}

            <!-- SUMMARY -->
            <tr>

                <td colspan="5" class="summary-footer-cell">

                    <div class="summary-container">

                        <div class="summary-item">
                            <span class="summary-label">Total Subjects:</span>
                            <span class="summary-value">{total_subjects}</span>
                        </div>

                        <div class="summary-item">
                            <span class="summary-label">Total Units:</span>
                            <span class="summary-value">{total_units:.1f}</span>
                        </div>

                        <div class="summary-item">
                            <span class="summary-label">Average Grade:</span>
                            <span class="summary-value">{average_grade:.2f}</span>
                        </div>

                        <div class="summary-item gwa-highlight">
                            <span>GWA:</span>
                            <span>{gwa:.4f}</span>
                        </div>

                    </div>

                </td>

            </tr>

        </tbody>

    </table>

</div>
"""

display(HTML(html))

eme

In [ ]:
import html as py_html
from IPython.display import display, HTML

# ===========================================================
# 1. STUDENT GRADE DATA
# Format: Code, Units, Teachers, Grade
# ===========================================================

students_data = {
    "Sheena Claire P. Galez": {
        "course": "BSCS", "year_level": 1,
        "subjects": [
            ("CSCC 101", 3, "Arianne Joy Cari-an", 2.25),
            ("CSCC 102", 3, "Brenda Mondejar Balala", 2.00),
            ("CSMATH 1", 3, "Victorino R. Tobias, Jr", 1.75),
            ("MST 111", 3, "Natalie Gallaron", 2.00),
            ("PE 1", 2, "Christian Dave Suacillo Ignacio", 1.25),
        ]
    },
    "Joshua Andre Bautista": {
        "course": "BSP", "year_level": 1,
        "subjects": [
            ("PSY 101", 3, "Paolo Mendoza", 1.25),
            ("PSY 102", 3, "Rowena Garcia", 1.00),
            ("PSY 103", 3, "Victor Ramos", 1.75),
            ("STAT 101", 3, "Elisa Dela Cruz", 1.00),
            ("GEC 109", 3, "Marvin Aquino", 1.00),
        ]
    },
    "Andrea Mae Castillo": {
        "course": "BSTM", "year_level": 1,
        "subjects": [
            ("TM 101", 3, "Carlo Reyes", 1.25),
            ("TM 102", 2, "Monica Flores", 2.25),
            ("TM 103", 3, "Benjamin Santos", 2.00),
            ("TM 104", 2, "Rachel Lopez", 2.00),
            ("GEC 101", 3, "Stephen Cruz", 1.00),
        ]
    },
    "Patricia Nicole Ramos": {
        "course": "BSA", "year_level": 1,
        "subjects": [
            ("ACC 101", 3, "Rafael Navarro", 1.25),
            ("ACC 102", 3, "Liza Fernandez", 1.00),
            ("MATH 103", 2, "Oscar Rivera", 2.00),
            ("TAX 101", 2, "Julia Castillo", 2.75),
            ("GEC 108", 2, "Dennis Villareal", 2.00),
        ]
    },
    "Camille Rose Villanueva": {
        "course": "BSC", "year_level": 1,
        "subjects": [
            ("CRM 101", 3, "Albert Reyes", 1.00),
            ("CRM 102", 3, "Grace Santos", 1.50),
            ("CRM 103", 3, "Francis Cruz", 2.00),
            ("SOC 101", 2, "Camille Flores", 2.00),
            ("GEC 106", 2, "Jerome Garcia", 2.00),
        ]
    },
    "Mark Joseph Dela Cruz": {
        "course": "BSBA", "year_level": 1,
        "subjects": [
            ("BA 101", 3, "Michael Torres", 1.25),
            ("BA 102", 3, "Christina Bautista", 2.00),
            ("ECON 101", 2, "Gabriel Navarro", 2.00),
            ("MKT 101", 2, "Samantha Castillo", 3.00),
            ("NSTP 1", 2, "Ronald Aquino", 1.00),
        ]
    },
    "Sofia Anne Garcia": {
        "course": "BSHM", "year_level": 1,
        "subjects": [
            ("HM 101", 3, "Roberto Garcia", 1.25),
            ("HM 102", 3, "Michelle Ramos", 1.00),
            ("HM 103", 3, "Anthony Mercado", 2.00),
            ("CUL 101", 2, "Patricia Gomez", 2.25),
            ("GEC 104", 2, "Joseph Fernandez", 1.75),
        ]
    },
    "Daniel Patrick Mendoza": {
        "course": "BSEd", "year_level": 1,
        "subjects": [
            ("EDU 101", 3, "Vanessa Aquino", 1.25),
            ("EDU 102", 3, "Mark Villanueva", 1.00),
            ("ENG 101", 3, "Teresa Mendoza", 2.00),
            ("GEC 105", 3, "Kevin Castillo", 1.00),
            ("PE 1", 2, "Denise Rivera", 2.75),
        ]
    },
    "John Mark Marcos": {
        "course": "BSME", "year_level": 1,
        "subjects": [
            ("PHY061", 3, "Ana Mercado", 1.25),
            ("CH012", 1, "Jonny Crzy", 1.00),
            ("MAT060", 4, "Mark Estela", 1.75),
            ("GEC104", 3, "Kim Ageste", 2.00),
            ("PED01", 2, "DJ Dela Cryz", 2.00),
        ]
    },
    "Gabriel Luis Fernandez": {
        "course": "BSN", "year_level": 1,
        "subjects": [
            ("NURS 101", 3, "Adrian Lopez", 1.25),
            ("NURS 102", 3, "Beatrice Ramos", 1.00),
            ("ANAT 101", 3, "Samuel Torres", 1.00),
            ("BIO 101", 3, "Katrina Bautista", 2.00),
            ("PE 1", 2, "Noel Mercado", 2.25),
        ]
    }
}

# ===========================================================
# 2. RENDER THE TABLES DYNAMICALLY WITH SELECTOR INTEGRATION
# ===========================================================

def safe_text(value):
    if value is None:
        return ""
    return py_html.escape(str(value))

reports_html = ""
select_options_html = "<option value=\"all\">All Students</option>"

for name, info in students_data.items():
    subjects = info["subjects"]
    total_subjects = len(subjects)

    total_units = 0
    weighted_grades_sum = 0
    valid_grade_units = 0
    grades_list = []

    # Generate options for the selection control dropdown
    select_options_html += f'<option value="{safe_text(name)}">{safe_text(name)}</option>'

    rows_markup = ""
    for i, subj in enumerate(subjects):
        code, units, teacher, grade = subj
        row_class = "even-row" if i % 2 == 0 else "odd-row"

        units_val = float(units) if units is not None else 0.0
        total_units += units_val

        if grade is not None:
            grade_val = float(grade)
            grades_list.append(grade_val)
            weighted_grades_sum += grade_val * units_val
            valid_grade_units += units_val
            grade_display = f"{grade_val:.2f}"
        else:
            grade_display = "N/A"

        units_display = f"{units_val:.1f}" if units is not None else ""

        rows_markup += f"""
        <tr class="{row_class}">
            <td>{safe_text(code)}</td>
            <td>{safe_text(code)} Description</td>
            <td>{safe_text(teacher)}</td>
            <td>{units_display}</td>
            <td>{grade_display}</td>
        </tr>
        """

    avg_grade = sum(grades_list) / len(grades_list) if grades_list else 0.0
    gwa = weighted_grades_sum / valid_grade_units if valid_grade_units > 0 else 0.0

    # Store student name in data-student attribute to filter dynamically
    reports_html += f"""
    <div class="student-report" data-student="{safe_text(name)}">
        <table class="unified-table">
            <colgroup>
                <col style="width: 15%;">
                <col style="width: 35%;">
                <col style="width: 25%;">
                <col style="width: 12%;">
                <col style="width: 13%;">
            </colgroup>
            <thead>
                <tr>
                    <th colspan="5" class="main-header">STUDENT GRADE REPORT</th>
                </tr>
                <tr>
                    <td colspan="5" class="student-info-cell">
                        <div class="info-grid">
                            <div class="info-item">
                                <span class="info-label">Student Name:</span>
                                <span>{safe_text(name)}</span>
                            </div>
                            <div class="info-item">
                                <span class="info-label">Course:</span>
                                <span>{safe_text(info['course'])}</span>
                            </div>
                            <div class="info-item">
                                <span class="info-label">Year Level:</span>
                                <span>{safe_text(info['year_level'])}</span>
                            </div>
                        </div>
                    </td>
                </tr>
                <tr class="col-header">
                    <th>Code</th>
                    <th>Description</th>
                    <th>Teachers</th>
                    <th>Units</th>
                    <th>Grade</th>
                </tr>
            </thead>
            <tbody>
                {rows_markup}
                <tr>
                    <td colspan="5" class="summary-footer-cell">
                        <div class="summary-container">
                            <div class="summary-item">
                                <span class="summary-label">Total Subjects:</span>
                                <span class="summary-value">{total_subjects}</span>
                            </div>
                            <div class="summary-item">
                                <span class="summary-label">Total Units:</span>
                                <span class="summary-value">{total_units:.1f}</span>
                            </div>
                            <div class="summary-item">
                                <span class="summary-label">Average Grade:</span>
                                <span class="summary-value">{avg_grade:.2f}</span>
                            </div>
                            <div class="summary-item gwa-highlight">
                                <span>GWA:</span>
                                <span>{gwa:.4f}</span>
                            </div>
                        </div>
                    </td>
                </tr>
            </tbody>
        </table>
    </div>
    """

# ===========================================================
# 3. COMPLETE CSS STYLESHEET AND LAYOUT
# ===========================================================

html_layout = f"""
<style>
* {{ box-sizing: border-box; }}

body {{
    margin: 0;
    padding: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #ffffff;
    color: #111111;
}}

.report-container {{
    width: 100%;
    max-width: 1200px;
    margin: 0 auto;
    padding: 15px;
    background: #ffffff;
}}

.btn-container {{
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 15px;
    margin-bottom: 25px;
    flex-wrap: wrap;
}}

.student-select {{
    background: #ffffff;
    color: #111111;
    border: 1px solid #111111;
    padding: 12px 20px;
    font-size: 15px;
    font-weight: bold;
    border-radius: 30px;
    cursor: pointer;
    outline: none;
    min-width: 220px;
}}

.print-btn {{
    background: #ffffff;
    color: #111111;
    border: 1px solid #111111;
    padding: 12px 25px;
    font-size: 16px;
    font-weight: bold;
    border-radius: 30px;
    cursor: pointer;
}}

.print-btn:hover, .student-select:hover {{
    background: #111111;
    color: #ffffff;
}}

.unified-table {{
    width: 100%;
    table-layout: fixed;
    border-collapse: collapse;
    border: 2px solid #111111;
    background: #ffffff;
}}

.main-header {{
    background: #ffffff;
    color: #111111;
    text-align: left !important;
    vertical-align: middle !important;
    font-size: 24px;
    font-weight: bold;
    padding: 20px;
    letter-spacing: 1.5px;
    border-bottom: 2px solid #111111;
}}

.student-info-cell {{
    background: #ffffff;
    padding: 15px 20px;
    border-bottom: 2px solid #111111;
    text-align: left !important;
    vertical-align: middle !important;
}}

.info-grid {{
    width: 100%;
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 15px;
    align-items: center;
    text-align: left;
}}

.info-item {{
    width: 100%;
    display: flex;
    justify-content: flex-start;
    align-items: center;
    gap: 8px;
    text-align: left;
}}

.info-label {{
    font-weight: bold;
    color: #111111;
}}

.col-header th {{
    background: #ffffff;
    color: #111111;
    padding: 12px 10px;
    font-weight: bold;
    font-size: 14px;
    text-align: left !important;
    vertical-align: middle !important;
    border-bottom: 2px solid #111111;
    border-right: 1px solid #cccccc;
}}

.col-header th:last-child {{
    border-right: none;
}}

.unified-table tbody td {{
    padding: 12px 10px;
    font-size: 14px;
    background-color: #ffffff;
    text-align: left !important;
    vertical-align: middle !important;
    border-bottom: 1px solid #dddddd;
    border-right: 1px solid #dddddd;
    word-wrap: break-word;
    overflow-wrap: break-word;
}}

.unified-table tbody td:last-child {{
    border-right: none;
}}

.even-row td, .odd-row td {{
    background-color: #ffffff !important;
}}

.unified-table tbody tr:hover td {{
    background-color: #f8f8f8 !important;
}}

.summary-footer-cell {{
    background: #ffffff;
    border-top: 2px solid #111111;
    padding: 18px 20px;
    text-align: left !important;
    vertical-align: middle !important;
}}

.summary-container {{
    width: 100%;
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 15px;
    align-items: center;
    text-align: left;
}}

.summary-item {{
    width: 100%;
    font-size: 15px;
    text-align: left;
    white-space: nowrap;
}}

.summary-label {{
    font-weight: bold;
    color: #111111;
}}

.summary-value {{
    font-weight: bold;
    color: #111111;
}}

.gwa-highlight {{
    background: #ffffff;
    color: #111111;
    border: 2px solid #111111;
    padding: 8px 18px;
    border-radius: 6px;
    font-weight: bold;
    display: flex;
    justify-content: flex-start;
    align-items: center;
    gap: 8px;
    width: fit-content;
}}

.student-report {{
    margin-bottom: 35px;
    display: block; /* Shown by default */
}}

@media print {{
    @page {{
        size: A4 landscape;
        margin: 8mm;
    }}

    html, body {{
        width: 100%;
        margin: 0 !important;
        padding: 0 !important;
        background: #ffffff !important;
    }}

    .no-print {{
        display: none !important;
    }}

    .report-container {{
        width: 100% !important;
        max-width: none !important;
        margin: 0 !important;
        padding: 0 !important;
    }}

    /* Ensure only currently visible filtered selections print, or all if selected */
    .student-report[style*="display: none"] {{
        display: none !important;
    }}

    .student-report {{
        margin: 0 !important;
        padding: 0 !important;
        break-after: page;
        page-break-after: always;
    }}

    .student-report:last-child {{
        break-after: auto;
        page-break-after: auto;
    }}

    .unified-table {{
        width: 100% !important;
        table-layout: fixed !important;
        border: 2px solid #111111 !important;
        border-collapse: collapse !important;
    }}

    .unified-table th, .unified-table td {{
        text-align: left !important;
        vertical-align: middle !important;
    }}

    .main-header, .student-info-cell {{
        text-align: left !important;
    }}

    .info-grid, .summary-container {{
        display: grid !important;
        grid-template-columns: repeat(3, 1fr) !important;
        text-align: left !important;
    }}

    .summary-container {{
        grid-template-columns: repeat(4, 1fr) !important;
    }}

    .info-item, .summary-item, .gwa-highlight {{
        justify-content: flex-start !important;
        text-align: left !important;
    }}

    tr {{
        page-break-inside: avoid !important;
    }}

    thead {{ display: table-header-group; }}
    tbody {{ display: table-row-group; }}
}}
</style>

<div class="report-container">
    <div class="btn-container no-print">
        <select class="student-select" id="studentSelector" onchange="filterStudentReport()">
            {select_options_html}
        </select>
        <button class="print-btn" onclick="var oldTitle = document.title; document.title = ''; window.print(); setTimeout(function() {{ document.title = oldTitle; }}, 100);">
            🖨️ Save / Print to PDF
        </button>
    </div>
    {reports_html}
</div>

<script>
function filterStudentReport() {{
    var selector = document.getElementById('studentSelector');
    var selectedVal = selector.value;
    var reports = document.querySelectorAll('.student-report');

    reports.forEach(function(report) {{
        if (selectedVal === 'all') {{
            report.style.display = 'block';
        }} else {{
            if (report.getAttribute('data-student') === selectedVal) {{
                report.style.display = 'block';
            }} else {{
                report.style.display = 'none';
            }}
        }}
    }});
}}
</script>
"""

display(HTML(html_layout))


may scoreboard na

In [22]:
import html as py_html
from IPython.display import display, HTML

# ===========================================================
# 1. STUDENT GRADE DATA
# ===========================================================

students_data = {
    "Sheena Claire P. Galez": {
        "course": "BSCS", "year_level": 1,
        "subjects": [
            ("CSCC 101", 3, "Arianne Joy Cari-an", 2.25),
            ("CSCC 102", 3, "Brenda Mondejar Balala", 2.00),
            ("CSMATH 1", 3, "Victorino R. Tobias, Jr", 1.75),
            ("MST 111", 3, "Natalie Gallaron", 2.00),
            ("PE 1", 2, "Christian Dave Suacillo Ignacio", 1.25),
        ]
    },
    "Joshua Andre Bautista": {
        "course": "BSP", "year_level": 1,
        "subjects": [
            ("PSY 101", 3, "Paolo Mendoza", 1.25),
            ("PSY 102", 3, "Rowena Garcia", 1.00),
            ("PSY 103", 3, "Victor Ramos", 1.75),
            ("STAT 101", 3, "Elisa Dela Cruz", 1.00),
            ("GEC 109", 3, "Marvin Aquino", 1.00),
        ]
    },
    "Andrea Mae Castillo": {
        "course": "BSTM", "year_level": 1,
        "subjects": [
            ("TM 101", 3, "Carlo Reyes", 1.25),
            ("TM 102", 2, "Monica Flores", 2.25),
            ("TM 103", 3, "Benjamin Santos", 2.00),
            ("TM 104", 2, "Rachel Lopez", 2.00),
            ("GEC 101", 3, "Stephen Cruz", 1.00),
        ]
    },
    "Patricia Nicole Ramos": {
        "course": "BSA", "year_level": 1,
        "subjects": [
            ("ACC 101", 3, "Rafael Navarro", 1.25),
            ("ACC 102", 3, "Liza Fernandez", 1.00),
            ("MATH 103", 2, "Oscar Rivera", 2.00),
            ("TAX 101", 2, "Julia Castillo", 2.75),
            ("GEC 108", 2, "Dennis Villareal", 2.00),
        ]
    },
    "Camille Rose Villanueva": {
        "course": "BSC", "year_level": 1,
        "subjects": [
            ("CRM 101", 3, "Albert Reyes", 1.00),
            ("CRM 102", 3, "Grace Santos", 1.50),
            ("CRM 103", 3, "Francis Cruz", 2.00),
            ("SOC 101", 2, "Camille Flores", 2.00),
            ("GEC 106", 2, "Jerome Garcia", 2.00),
        ]
    },
    "Mark Joseph Dela Cruz": {
        "course": "BSBA", "year_level": 1,
        "subjects": [
            ("BA 101", 3, "Michael Torres", 1.25),
            ("BA 102", 3, "Christina Bautista", 2.00),
            ("ECON 101", 2, "Gabriel Navarro", 2.00),
            ("MKT 101", 2, "Samantha Castillo", 3.00),
            ("NSTP 1", 2, "Ronald Aquino", 1.00),
        ]
    },
    "Sofia Anne Garcia": {
        "course": "BSHM", "year_level": 1,
        "subjects": [
            ("HM 101", 3, "Roberto Garcia", 1.25),
            ("HM 102", 3, "Michelle Ramos", 1.00),
            ("HM 103", 3, "Anthony Mercado", 2.00),
            ("CUL 101", 2, "Patricia Gomez", 2.25),
            ("GEC 104", 2, "Joseph Fernandez", 1.75),
        ]
    },
    "Daniel Patrick Mendoza": {
        "course": "BSEd", "year_level": 1,
        "subjects": [
            ("EDU 101", 3, "Vanessa Aquino", 1.25),
            ("EDU 102", 3, "Mark Villanueva", 1.00),
            ("ENG 101", 3, "Teresa Mendoza", 2.00),
            ("GEC 105", 3, "Kevin Castillo", 1.00),
            ("PE 1", 2, "Denise Rivera", 2.75),
        ]
    },
    "John Mark Marcos": {
        "course": "BSME", "year_level": 1,
        "subjects": [
            ("PHY061", 3, "Ana Mercado", 1.25),
            ("CH012", 1, "Jonny Crzy", 1.00),
            ("MAT060", 4, "Mark Estela", 1.75),
            ("GEC104", 3, "Kim Ageste", 2.00),
            ("PED01", 2, "DJ Dela Cryz", 2.00),
        ]
    },
    "Gabriel Luis Fernandez": {
        "course": "BSN", "year_level": 1,
        "subjects": [
            ("NURS 101", 3, "Adrian Lopez", 1.25),
            ("NURS 102", 3, "Beatrice Ramos", 1.00),
            ("ANAT 101", 3, "Samuel Torres", 1.00),
            ("BIO 101", 3, "Katrina Bautista", 2.00),
            ("PE 1", 2, "Noel Mercado", 2.25),
        ]
    }
}

def safe_text(value):
    return "" if value is None else py_html.escape(str(value))

# ===========================================================
# 2. YEAR LEVEL LABELS
# ===========================================================

year_names = {
    1: "First Year",
    2: "Second Year",
    3: "Third Year",
    4: "Fourth Year",
    5: "Fifth Year"
}

# ===========================================================
# 3. CALCULATE GWA AND CREATE SCOREBOARD
# ===========================================================

scoreboard_data = []

for name, info in students_data.items():
    weighted_sum = 0.0
    valid_units = 0.0

    for code, units, teacher, grade in info["subjects"]:
        if grade is not None and units is not None:
            weighted_sum += float(grade) * float(units)
            valid_units += float(units)

    gwa = weighted_sum / valid_units if valid_units > 0 else None
    status = (
        "NO GRADE" if gwa is None
        else "PASSED" if gwa <= 3.00
        else "FAILED"
    )

    scoreboard_data.append({
        "name": name,
        "course": info["course"],
        "year_level": info["year_level"],
        "gwa": gwa,
        "status": status
    })

# Sort in ascending order so the lowest numerical value (highest academic grade) is Rank 1
scoreboard_data.sort(
    key=lambda s: (
        s["gwa"] is None,
        s["gwa"] if s["gwa"] is not None else float('inf')
    )
)

scoreboard_rows = ""
for rank, student in enumerate(scoreboard_data, start=1):
    gwa = student["gwa"]
    gwa_display = f"{gwa:.4f}" if gwa is not None else "N/A"
    status_class = student["status"].lower().replace(" ", "-")
    year_display = year_names.get(
        student["year_level"],
        f"Year {student['year_level']}"
    )

    scoreboard_rows += f"""
    <tr>
        <td class="rank">{rank}</td>
        <td>{safe_text(student['name'])}</td>
        <td>{safe_text(student['course'])}</td>
        <td>{safe_text(year_display)}</td>
        <td class="gwa">{gwa_display}</td>
        <td><span class="status {status_class}">{student['status']}</span></td>
    </tr>
    """

scoreboard_html = f"""
<div class="scoreboard">
    <h2>GPA SCOREBOARD</h2>
    <p class="subtitle">Ranked from highest academic grade (lowest numerical GWA) to lowest</p>
    <div class="scoreboard-scroll">
        <table class="scoreboard-table">
            <colgroup>
                <col style="width: 8%;">
                <col style="width: 32%;">
                <col style="width: 13%;">
                <col style="width: 17%;">
                <col style="width: 16%;">
                <col style="width: 16%;">
            </colgroup>
            <thead>
                <tr>
                    <th>Rank</th>
                    <th>Student Name</th>
                    <th>Course</th>
                    <th>Year Level</th>
                    <th>GPA (GWA)</th>
                    <th>Status</th>
                </tr>
            </thead>
            <tbody>{scoreboard_rows}</tbody>
        </table>
    </div>
</div>
"""

# ===========================================================
# 4. GENERATE INDIVIDUAL STUDENT GRADE REPORTS
# ===========================================================

reports_html = ""
select_options_html = '<option value="all">All Students</option>'

for name, info in students_data.items():
    subjects = info["subjects"]
    total_subjects = len(subjects)
    total_units = 0.0
    weighted_sum = 0.0
    valid_grade_units = 0.0
    grades_list = []

    select_options_html += (
        f'<option value="{safe_text(name)}">{safe_text(name)}</option>'
    )

    rows_markup = ""
    for i, (code, units, teacher, grade) in enumerate(subjects):
        row_class = "even-row" if i % 2 == 0 else "odd-row"
        units_val = float(units) if units is not None else 0.0
        total_units += units_val

        if grade is not None:
            grade_val = float(grade)
            grades_list.append(grade_val)
            weighted_sum += grade_val * units_val
            valid_grade_units += units_val
            grade_display = f"{grade_val:.2f}"
        else:
            grade_display = "N/A"

        units_display = f"{units_val:.1f}" if units is not None else ""

        rows_markup += f"""
        <tr class="{row_class}">
            <td>{safe_text(code)}</td>
            <td>{safe_text(teacher)}</td>
            <td>{units_display}</td>
            <td>{grade_display}</td>
        </tr>
        """

    avg_grade = (
        sum(grades_list) / len(grades_list)
        if grades_list else 0.0
    )
    gwa = weighted_sum / valid_grade_units if valid_grade_units else 0.0

    year_display = year_names.get(
        info["year_level"],
        f"Year {info['year_level']}"
    )

    reports_html += f"""
    <div class="student-report" data-student="{safe_text(name)}">
        <table class="unified-table">
            <colgroup>
                <col style="width:25%">
                <col style="width:47%">
                <col style="width:12%">
                <col style="width:16%">
            </colgroup>
            <thead>
                <tr>
                    <th colspan="4" class="main-header">STUDENT GRADE REPORT</th>
                </tr>
                <tr>
                    <td colspan="4" class="student-info-cell">
                        <div class="info-grid">
                            <div><b>Student Name:</b> {safe_text(name)}</div>
                            <div><b>Course:</b> {safe_text(info["course"])}</div>
                            <div><b>Year Level:</b> {safe_text(year_display)}</div>
                        </div>
                    </td>
                </tr>
                <tr class="col-header">
                    <th>Subject</th>
                    <th>Teachers</th>
                    <th>Units</th>
                    <th>Grade</th>
                </tr>
            </thead>
            <tbody>
                {rows_markup}
                <tr>
                    <td colspan="4" class="summary-footer-cell">
                        <div class="summary-container">
                            <div><b>Total Subjects:</b> {total_subjects}</div>
                            <div><b>Total Units:</b> {total_units:.1f}</div>
                            <div><b>Average Grade:</b> {avg_grade:.2f}</div>
                            <div class="gwa-highlight"><b>GWA:</b> {gwa:.4f}</div>
                        </div>
                    </td>
                </tr>
            </tbody>
        </table>
    </div>
    """

# ===========================================================
# 5. CSS STYLES AND PAGE LAYOUT
# ===========================================================

html_layout = f"""
<style>
* {{ box-sizing: border-box; }}

body {{
    margin: 0;
    padding: 0;
    font-family: Arial, Helvetica, sans-serif;
    color: #111;
    background: white;
}}

.report-container {{
    width: 100%;
    max-width: 760px;
    margin: auto;
    padding: 15px;
}}

.btn-container {{
    display: flex;
    justify-content: center;
    align-items: center;
    flex-wrap: wrap;
    gap: 15px;
    margin-bottom: 25px;
}}

.student-select, .print-btn {{
    background: white;
    color: #111;
    border: 1px solid #111;
    padding: 8px 16px;
    font-size: 13px;
    font-weight: bold;
    border-radius: 30px;
    cursor: pointer;
}}

.student-select {{ min-width: 220px; }}

.student-select:hover, .print-btn:hover {{
    background: #111;
    color: white;
}}

.scoreboard {{
    margin-bottom: 25px;
    border: 2px solid #111;
    padding: 15px;
    background: white;
}}

.scoreboard h2 {{
    margin: 0 0 4px;
    font-size: 20px;
    letter-spacing: 1px;
    text-align: center;
}}

.subtitle {{
    margin: 0 0 14px;
    color: #555;
    font-size: 12px;
    text-align: center;
}}

.scoreboard-scroll {{
    width: 100%;
    overflow-x: auto;
}}

.scoreboard-table {{
    width: 100%;
    border-collapse: collapse;
    table-layout: fixed;
}}

.scoreboard-table th {{
    padding: 8px 5px;
    background: #111;
    color: white;
    text-align: center;
    border: 1px solid #444;
    overflow-wrap: break-word;
    font-size: 12px;
}}

.scoreboard-table td {{
    padding: 8px 5px;
    border: 1px solid #ddd;
    overflow-wrap: anywhere;
    font-size: 12px;
    text-align: center;
}}

.scoreboard-table tbody tr:nth-child(even) {{ background: #f4f4f4; }}
.scoreboard-table .rank {{ font-weight: bold; }}
.gwa {{ font-weight: bold; white-space: nowrap; }}
.status {{ font-weight: bold; }}
.status.passed {{ color: #166534; }}
.status.failed {{ color: #b91c1c; }}
.status.no-grade {{ color: #555; }}

.unified-table {{
    width: 100%;
    table-layout: fixed;
    border-collapse: collapse;
    border: 2px solid #111;
    margin-bottom: 0;
}}

.main-header {{
    text-align: center !important;
    font-size: 20px;
    padding: 12px;
    border-bottom: 2px solid #111;
}}

.student-info-cell {{
    padding: 10px 12px;
    border-bottom: 2px solid #111;
}}

.info-grid {{
    display: flex;
    justify-content: space-around;
    gap: 15px;
    font-size: 13px;
}}

.col-header th {{
    padding: 8px 6px;
    text-align: center !important;
    border-bottom: 2px solid #111;
    border-right: 1px solid #ddd;
    font-size: 12px;
}}
.col-header th:last-child {{ border-right: none; }}

.unified-table tbody td {{
    padding: 8px 6px;
    font-size: 12px;
    border-bottom: 1px solid #ddd;
    border-right: 1px solid #ddd;
    overflow-wrap: anywhere;
    text-align: center !important;
}}
.unified-table tbody td:last-child {{ border-right: none; }}

.even-row td, .odd-row td {{ background-color: #ffffff !important; }}
.unified-table tbody tr:hover td {{ background-color: #fcfcfc !important; }}
.summary-footer-cell {{ padding: 12px; border-top: 2px solid #111; }}

.summary-container {{
    display: flex;
    justify-content: space-around;
    align-items: center;
    font-size: 13px;
}}

.gwa-highlight {{
    border: 2px solid #111;
    padding: 4px 10px;
    border-radius: 6px;
    font-weight: bold;
}}

.student-report {{ margin-bottom: 25px; }}

@media (max-width: 650px) {{
    .report-container {{ padding: 6px; }}
    .scoreboard {{ padding: 8px; }}
    .scoreboard-table th, .scoreboard-table td {{
        padding: 5px 3px;
        font-size: 10px;
    }}
    .info-grid {{ flex-direction: column; align-items: center; gap: 5px; }}
    .summary-container {{ flex-direction: column; gap: 6px; }}
    .unified-table tbody td, .col-header th {{
        padding: 5px 3px;
        font-size: 10px;
    }}
}}

@media print {{
    @page {{ size: A4 landscape; margin: 8mm; }}
    .no-print {{ display: none !important; }}
    .report-container {{
        max-width: none;
        margin: 0;
        padding: 0;
    }}
    .scoreboard {{
        break-after: page;
        page-break-after: always;
    }}
    .scoreboard-scroll {{ overflow: visible; }}
    .scoreboard-table th {{
        background: #eee !important;
        color: #111 !important;
        print-color-adjust: exact;
        -webkit-print-color-adjust: exact;
    }}
    .student-report[style*="display: none"] {{ display: none !important; }}
    .student-report {{
        break-after: page;
        page-break-after: always;
        margin: 0;
    }}
    .student-report:last-child {{ break-after: auto; page-break-after: auto; }}
    tr {{ break-inside: avoid; page-break-inside: avoid; }}
    thead {{ display: table-header-group; }}
    tbody {{ display: table-row-group; }}
}}
</style>

<div class="report-container">
    {scoreboard_html}

    <div class="btn-container no-print">
        <select class="student-select" id="studentSelector" onchange="filterStudentReport()">
            {select_options_html}
        </select>
        <button class="print-btn" onclick="window.print()">
            Save / Print to PDF
        </button>
    </div>

    {reports_html}
</div>

<script>
function filterStudentReport() {{
    var selectedVal = document.getElementById('studentSelector').value;
    var reports = document.querySelectorAll('.student-report');

    reports.forEach(function(report) {{
        report.style.display =
            (selectedVal === 'all' || report.getAttribute('data-student') === selectedVal)
            ? 'block' : 'none';
    }});
}}
</script>
"""

display(HTML(html_layout))


Rank,Student Name,Course,Year Level,GPA (GWA),Status
1,Joshua Andre Bautista,BSP,First Year,1.2000,PASSED
2,Gabriel Luis Fernandez,BSN,First Year,1.4464,PASSED
3,Daniel Patrick Mendoza,BSEd,First Year,1.5179,PASSED
4,Sofia Anne Garcia,BSHM,First Year,1.5962,PASSED
5,Andrea Mae Castillo,BSTM,First Year,1.6346,PASSED
6,Camille Rose Villanueva,BSC,First Year,1.6538,PASSED
7,John Mark Marcos,BSME,First Year,1.6731,PASSED
8,Patricia Nicole Ramos,BSA,First Year,1.6875,PASSED
9,Mark Joseph Dela Cruz,BSBA,First Year,1.8125,PASSED
10,Sheena Claire P. Galez,BSCS,First Year,1.8929,PASSED
